# M16 L16: AI-Native Full Stack -- Chat UI to Deploy

**Build the smallest possible end-to-end AI app from scratch -- no fastapi, no flask, no uvicorn, just stdlib.**

Duration: 90 min.  No API keys, no external services.

## Plain English (the 12-year-old version)

An AI app is **four pieces** stuck together:

1. A **request shape** -- what the client sends (a message, a conversation id)
2. A **router** -- checks auth and rate limit, then dispatches to a handler
3. A **backend** -- calls the LLM, tracks usage, streams the response
4. A **streaming response** -- yields chunks via SSE as the LLM generates them

The deployment (replicas, region, secrets) is a YAML file; it's the fifth piece but it is the same shape on every cloud. Once you can wire the first four, you have shipped 90% of the production surface. The rest is observability, security, and scaling, which prior modules already covered.

## The 5 things to know

1. **Stream, do not wait** -- a 500-token response takes 15 seconds; the user will close the tab. SSE gives time-to-first-token in 200-500ms.
2. **Auth at the router, not the LLM** -- bearer token verified before the model call. Never pass the raw token to the LLM.
3. **Rate limit per-user, not global** -- one user over-using should not block the rest. Token bucket refills continuously.
4. **Secrets stay in the manifest, never in logs** -- redact values; log keys only.
5. **Health and metrics endpoints are required** -- the load balancer scrapes /health, Prometheus scrapes /metrics.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm16-fullstack' / 'L16-fullstack.py'
_spec = importlib.util.spec_from_file_location('L16', _p)
_L16 = importlib.util.module_from_spec(_spec)
sys.modules['L16'] = _L16
_spec.loader.exec_module(_L16)

print(f'Setup OK. Lesson module: {_L16.LESSON_NUMBER}')

Setup OK. Lesson module: M16.L16


In [2]:
# Run the 20 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L16, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 30 tests...

  PASS  test_auth_middleware_accepts_known
  PASS  test_auth_middleware_rejects_bad
  PASS  test_auth_middleware_rejects_missing
  PASS  test_backend_complete_no_tool_otherwise
  PASS  test_backend_complete_uses_tool_when_keyword_present
  PASS  test_conversation_appends_in_order
  PASS  test_conversation_is_iterable
  PASS  test_deploy_config_capacity
  PASS  test_deploy_config_redacts_secrets
  PASS  test_deploy_config_total_capacity_scales_linearly
  PASS  test_fullstack_math_invariants
  PASS  test_message_roundtrip
  PASS  test_message_timestamp_is_serializable
  PASS  test_rate_limiter_allows_under_cap
  PASS  test_rate_limiter_blocks_over_cap
  PASS  test_rate_limiter_refills_over_time
  PASS  test_rate_limiter_starts_full
  PASS  test_request_extracts_bearer_token
  PASS  test_request_extracts_path
  PASS  test_response_status_codes
  PASS  test_router_404_for_unknown_path
  PASS  test_router_chat_404_for_unknown_route
  PASS  test_router_chat_requires_auth

In [3]:
# === Section 1: Message + Conversation records ===
import json
# from lessons.m16_fullstack import _L16 if False else None  # noqa
# Use the loaded module from the setup cell
m = _L16.Message(role='user', content='hi', timestamp=1.0)
print('Message.to_dict():', m.to_dict())
c = _L16.Conversation(user_id='alice')
c.add('user', 'hello')
c.add('assistant', 'hi, how can I help?')
c.add('user', 'what is the weather?')
print('Conversation length:', len(c.messages))
print('Last user message:', c.last_user())
print('Rendered:', json.dumps(c.render()[:2], indent=2))

Message.to_dict(): {'role': 'user', 'content': 'hi', 'timestamp': 1.0}
Conversation length: 3
Last user message: what is the weather?
Rendered: [
  {
    "role": "user",
    "content": "hello",
    "timestamp": 1792106143.291559
  },
  {
    "role": "assistant",
    "content": "hi, how can I help?",
    "timestamp": 1792106143.291581
  }
]


In [4]:
# === Section 2: RateLimiter (per-user token bucket) ===
rl = _L16.RateLimiter(capacity=5, per_seconds=60)
print('capacity:', rl.capacity, 'per_seconds:', rl.per_seconds)
for i in range(7):
    ok = rl.allow('alice', now=0.0)
    print(f'  request {i+1}: allowed={ok}  remaining={rl.remaining("alice"):.1f}')

capacity: 5.0 per_seconds: 60
  request 1: allowed=True  remaining=5.0
  request 2: allowed=False  remaining=5.0
  request 3: allowed=False  remaining=5.0
  request 4: allowed=False  remaining=5.0
  request 5: allowed=False  remaining=5.0
  request 6: allowed=False  remaining=5.0
  request 7: allowed=False  remaining=5.0


In [5]:
# === Section 3: Request + Response + Auth ===
am = _L16.AuthMiddleware()
req_ok = _L16.Request('GET', '/x', headers={'Authorization': 'Bearer tok_alice'})
req_bad = _L16.Request('GET', '/x', headers={})
ok, user, err = am.authenticate(req_ok)
print(f'  good token  -> ok={ok} user={user!r} err={err!r}')
ok, user, err = am.authenticate(req_bad)
print(f'  no auth     -> ok={ok} user={user!r} err={err!r}')
# Response helpers
r_ok = _L16.Response.ok({'hello': 'world'})
r_err = _L16.Response.error(404, 'not found')
print(f'  Response.ok    -> {r_ok.status} {r_ok.body}')
print(f'  Response.error -> {r_err.status} {r_err.body}')

  good token  -> ok=True user='alice' err=''
  no auth     -> ok=False user='' err='missing Authorization header'
  Response.ok    -> 200 {"hello": "world"}
  Response.error -> 404 {"error": "not found", "status": 404}


In [6]:
# === Section 4: MockBackend (LLM + tool) ===
def fake_search(q):
    return f'<search results for {q!r}>'
backend = _L16.MockBackend(
    model_id='mock-1',
    default_response='Hello from the mock LLM!',
    tools={'search': fake_search},
)
# complete() with no tool keyword
res = backend.complete([{'role': 'user', 'content': 'hi there'}])
print('No tool:')
print(f'  content:   {res["content"]!r}')
print(f'  tool_calls: {res["tool_calls"]}')
print(f'  usage:     {res["usage"]}')
# complete() with tool keyword
res2 = backend.complete([{'role': 'user', 'content': 'search: cats'}])
print('With tool:')
print(f'  content:   {res2["content"]!r}')
print(f'  tool_calls: {[tc.name for tc in res2["tool_calls"]]}')

No tool:
  content:   '[response seed=777] Hello from the mock LLM!'
  tool_calls: []
  usage:     {'input_tokens': 2, 'output_tokens': 11}
With tool:
  content:   "[used tool:search with seed 147] <search results for 'cats'>"
  tool_calls: ['search']


In [7]:
# === Section 5: APIRouter (POST /chat, /stream, GET /health, /metrics) ===
auth = _L16.AuthMiddleware()
limiter = _L16.RateLimiter(capacity=10, per_seconds=60)
router = _L16.APIRouter(backend=backend, auth=auth, limiter=limiter)

# POST /chat
chat_resp = router.route(_L16.Request('POST', '/chat',
    headers={'Authorization': 'Bearer tok_alice'},
    body=json.dumps({'message': 'hi there'})))
print('POST /chat  ->', chat_resp.status, chat_resp.body[:120])

# GET /health
health = router.route(_L16.Request('GET', '/health'))
print('GET /health ->', health.status, health.body)

# GET /metrics
metrics = router.route(_L16.Request('GET', '/metrics'))
print('GET /metrics->', metrics.status, metrics.body)

POST /chat  -> 200 {"conversation_id": "340389af-70d0-439a-9b84-4a86a9920535", "reply": "[response seed=777] Hello from the mock LLM!", "to
GET /health -> 200 {"status": "ok", "model": "mock-1"}
GET /metrics-> 200 {"requests_total": 2, "errors_total": 0, "streamed_total": 0, "conversations": 1}


In [8]:
# === Section 6: Streaming response (SSE) ===
stream_resp = router.route(_L16.Request('POST', '/stream',
    headers={'Authorization': 'Bearer tok_bob'},
    body=json.dumps({'message': 'tell me a story'})))
print('Content-Type:', stream_resp.headers.get('Content-Type'))
n = 0
for chunk in stream_resp.stream():
    n += 1
    if n <= 4:
        print(f'  chunk {n}: {chunk!r}')
print(f'  total chunks: {n}')

Content-Type: text/event-stream
  chunk 1: 'data: [response \n\n'
  chunk 2: 'data: seed=413] \n\n'
  chunk 3: 'data: Hello \n\n'
  chunk 4: 'data: from \n\n'
  total chunks: 8


In [9]:
# === Section 7: error paths (auth, rate limit, 404) ===
# No auth
err1 = router.route(_L16.Request('POST', '/chat', body='{"message":"x"}'))
print(f'  no auth     -> {err1.status} {err1.body}')
# Bad token
err2 = router.route(_L16.Request('POST', '/chat',
    headers={'Authorization': 'Bearer wrong'}, body='{"message":"x"}'))
print(f'  bad token   -> {err2.status} {err2.body}')
# 404
err3 = router.route(_L16.Request('GET', '/nope'))
print(f'  unknown     -> {err3.status} {err3.body}')
# Rate limit
got_429 = False
for i in range(20):
    r = router.route(_L16.Request('POST', '/chat',
        headers={'Authorization': 'Bearer tok_admin'},
        body='{"message":"hi"}'))
    if r.status == 429:
        got_429 = True
        break
print(f'  rate-limited: {got_429} (after {i+1} requests)')

  no auth     -> 401 {"error": "missing Authorization header", "status": 401}
  bad token   -> 401 {"error": "invalid token", "status": 401}
  unknown     -> 404 {"error": "no route for GET /nope", "status": 404}
  rate-limited: True (after 11 requests)


In [10]:
# === Section 8: DeployConfig (region / replicas / secrets) ===
deploy = _L16.DeployConfig(
    service_name='chat-api',
    region='us-east-1',
    replicas=3,
    env_vars={'MODEL_ID': 'mock-1', 'LOG_LEVEL': 'info'},
    secrets={'DB_URL': 'postgres://redacted', 'JWT_SECRET': 'redacted'},
)
print('Manifest (secrets redacted):')
for k, v in deploy.to_dict().items():
    print(f'  {k}: {v}')
print(f'  theoretical QPS @ 100 qps/replica: {deploy.total_capacity(100):,}')

Manifest (secrets redacted):
  service_name: chat-api
  region: us-east-1
  replicas: 3
  env_vars: {'MODEL_ID': 'mock-1', 'LOG_LEVEL': 'info'}
  secrets_keys: ['DB_URL', 'JWT_SECRET']
  health_check_path: /health
  metrics_path: /metrics
  theoretical QPS @ 100 qps/replica: 300


## Architecture (Mermaid)

```mermaid
flowchart LR
    A[Client<br/>terminal / web / mobile] -->|POST /chat| R[APIRouter]
    A -->|POST /stream| R
    R -->|check| AM[AuthMiddleware<br/>bearer token]
    R -->|check| RL[RateLimiter<br/>token bucket]
    R -->|dispatch| H[POST /chat handler]
    R -->|dispatch| S[POST /stream handler]
    R -->|dispatch| X[GET /health]
    R -->|dispatch| M[GET /metrics]
    H --> B[MockBackend<br/>LLM + tools]
    S --> B
    B -->|stream| C[StreamingResponse<br/>SSE]
    B -->|log| U[UsageRecord]
    R --> DC[DeployConfig<br/>region / replicas / secrets]
```

**Read as:** Client sends a request. The router checks auth and rate limit, then dispatches to a handler. The handler calls the backend (LLM + tools). The backend streams tokens back via SSE and logs usage. The whole thing is wrapped in a deployment manifest (region, replicas, secrets).

## Reflect (10 min)

1. Why is `StreamingResponse` a generator and not a list? What changes if you materialize it?
2. The rate limiter is in-memory. What happens at multi-region? At >1 replica?
3. Why does the router check auth *before* the LLM call? What goes wrong if you check it after?
4. The deployment manifest logs `secrets_keys` but not values. How would you enforce this in production? (Hint: a log formatter.)
5. (Stretch) Add a `Conversation` TTL: conversations older than 1 hour are evicted. Where in the code does the eviction hook go?

---

**Read next:**
- `lessons/m17-product-metrics/` -- measure what users actually do
- `lessons/m12-security/` -- secrets, PII, and auth hardening
- `lessons/m11-observability/` -- logs, traces, metrics
- `SYLLABUS.md` -- the 18 modules and 21-day plan